In [28]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, Callable


## Test Zone

In [19]:
"""Reading data"""
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet")
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [27]:
#training_data.info()
print(training_data.head(5))
#training_data["sensor"]
#sensors["sensor"]
#test_data.sample(15)
#sensors.info()
#sensors.query("coor_z == 0") #tous =0
sensors.head(5)




  sensor       time        power  temperature
0   N102        0.0  1487.964722    17.514429
1   N102   864000.0  1487.288818    17.820795
2   N102  1728000.0  1486.612915    17.573187
3   N102  2592000.0  1485.936890    16.513235
4   N102  3456000.0  1485.260986    16.303427


,sensor,coor_x,coor_y,coor_z
0,N2,0.5,0.0,0.0
1,N4,1.4,0.0,0.0
2,N5,0.5,2.4,0.0
3,N6,0.0,2.4,0.0
4,N7,0.0,3.5,0.0


In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="power", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
a = sampleP["power"].diff(periods=864000)

a.head(20)
power1 = training_data.query("power == 500.0000 or power == 1000.0000 or power == 1500.0000")
power1.plot.scatter(x="time", y="power", alpha=0.5)


In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = training_data.query(" sensor=='N418'")
sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="power", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)
c=b["power"].pct_change(periods=1)


In [ ]:
sampleN418b = sampleN418.sample(1000)
sampleN418b.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418b.plot.scatter(x="time", y="temperature", alpha=0.5)


## I. First Model

### 1. Distance Metrics

In [ ]:
def man_dist(sample_coo: np.ndarray, X: np.ndarray) -> np.ndarray:

    """Computes the Manhattan distance between a sample and the sensors.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the sensors' coordinates, of shape (N, D)
    Returns:
        dist: Distances of shape (N,)
    """

    return np.abs(X - sample_coo).sum(axis = 1)
   


def eucli_dist(sample_coo: np.ndarray, X: np.ndarray) -> np.ndarray:

    """Computes the Euclidean distance between a sample and the sensors.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the sensors' coordinates, of shape (N, D)
    Returns:
        dist: Distances of shape (N,)
    """ 
   
    return np.sqrt(((X - sample_coo) ** 2).sum(axis = 1))

### 2. Distance Score

In [ ]:
def distance_score(sample_coo: np.ndarray,
                   X: np.ndarray,
                   distance_fn: Callable = eucli_dist,
                   score_parameter: float = 1) -> np.ndarray:
    
    """Gives a score to each given sensor, depending on their distance to sample.
    The closer is the sample, the higher the score. We set an upper limit to it
    to avoid giving "blind confidence" to "very close" sensors.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the sensors' coordinates, of shape (N, D)
        function: Distance function chosen
        score_parameter: Hyper-parameter, set to 1 by default
    Returns:
        dist: Distances of shape (N,)
    """

    distance = distance_fn(sample_coo, X)
    score = 1 / (score_parameter*distance + 1)
    
    return score


### 3. Missing Values manager

### 4. K-Nearest neighbors

In [ ]:
def find_nearest_neighbors(
    sample_coo: np.ndarray, 
    X: np.ndarray, 
    distance_fn: Callable = eucli_dist, 
    k: int = 7):
    
    """Finds the indices of the k-Nearest Neighbors to a sample
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the the sensors' coordinates, of shape (N, D)
        distance_fn: Distance function
        k: Number of nearest neighbors taken, set to 7 by default

    Returns:
        indices: Neighbor indices of shape (k, )
    """

    distances = distance_fn(sample_coo, X)
    neighbor_indices = np.argsort(distances)[:k]

    
    return neighbor_indices